In [0]:
# Test complet de l'import météo pour toutes les zones
import sys
from pathlib import Path
import pandas as pd
import requests

# Configuration du projet
PROJECT_ROOT = Path("/Workspace/Users/n.jouglet23@gmail.com/energy_forecast")
CONFIG_DIR = PROJECT_ROOT / "config"

if str(CONFIG_DIR) not in sys.path:
    sys.path.append(str(CONFIG_DIR))

from zones_config import WEATHER_ZONES

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_gusts_10m",
    "cloud_cover",
    "surface_pressure",
]

# URLs pour Open-Meteo
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

# Configuration de la fenêtre de prédiction
TIMEZONE = "America/Toronto"
current_time = pd.Timestamp.now(tz=TIMEZONE).floor("h").tz_localize(None)
prediction_start = current_time.replace(hour=8)
if current_time.hour >= 8:
    prediction_start += pd.Timedelta(days=1)
prediction_end = prediction_start + pd.Timedelta(hours=24)

print("=" * 80)
print("TEST D'IMPORT MÉTÉO - TOUTES LES ZONES")
print("=" * 80)
print(f"\nNombre de zones configurées: {len(WEATHER_ZONES)}")
print(f"Fenêtre de prédiction: {prediction_start} → {prediction_end}")
print(f"Durée: 25 heures")

# Déterminer maintenant pour séparer historique/forecast
now = pd.Timestamp.now(tz=TIMEZONE).floor("h").tz_localize(None)
print(f"Heure actuelle: {now}")

# Test d'import pour chaque zone
success_count = 0
error_count = 0
zone_data = {}

for zone_key, zone_config in WEATHER_ZONES.items():
    print(f"\n{'─' * 80}")
    print(f"Zone: {zone_key.upper()} ({zone_config.city})")
    print(f"  Coordonnées: {zone_config.latitude:.4f}°N, {zone_config.longitude:.4f}°W")
    print(f"  Poids: {zone_config.weight:.4f}")
    
    all_data = []
    
    # ===== PARTIE 1: HISTORIQUE (prediction_start → now) =====
    # On récupère au moins 388h d'historique avant prediction_start pour calculer les features
    archive_start = prediction_start - pd.Timedelta(hours=388)
    if archive_start < now:
        print(f"  → Archive API: {archive_start} → {now} (dont 388h avant prédiction pour features)")
        archive_params = {
            "latitude": zone_config.latitude,
            "longitude": zone_config.longitude,
            "start_date": archive_start.strftime("%Y-%m-%d"),
            "end_date": now.strftime("%Y-%m-%d"),
            "hourly": ",".join(HOURLY_VARIABLES),
            "timezone": TIMEZONE,
            "temperature_unit": "celsius",
            "wind_speed_unit": "kmh",
            "precipitation_unit": "mm",
        }
        
        try:
            response = requests.get(ARCHIVE_URL, params=archive_params, timeout=60)
            response.raise_for_status()
            archive_data = response.json()
            
            if "hourly" in archive_data and "time" in archive_data["hourly"]:
                archive_df = pd.DataFrame(archive_data["hourly"])
                archive_df["time"] = pd.to_datetime(archive_df["time"])
                archive_df = archive_df[
                    (archive_df["time"] >= archive_start) & 
                    (archive_df["time"] < now)
                ]
                all_data.append(archive_df)
                print(f"    ✓ Archive: {len(archive_df)} heures")
            else:
                print(f"    ✗ Archive: structure invalide")
                
        except Exception as e:
            print(f"    ✗ Archive: {e}")
    
    # ===== PARTIE 2: FORECAST (now → prediction_end) =====
    if prediction_end > now:
        print(f"  → Forecast API: {now} → {prediction_end}")
        forecast_params = {
            "latitude": zone_config.latitude,
            "longitude": zone_config.longitude,
            "hourly": ",".join(HOURLY_VARIABLES),
            "timezone": TIMEZONE,
            "temperature_unit": "celsius",
            "wind_speed_unit": "kmh",
            "precipitation_unit": "mm",
        }
        
        try:
            response = requests.get(FORECAST_URL, params=forecast_params, timeout=60)
            response.raise_for_status()
            forecast_data = response.json()
            
            if "hourly" in forecast_data and "time" in forecast_data["hourly"]:
                forecast_df = pd.DataFrame(forecast_data["hourly"])
                forecast_df["time"] = pd.to_datetime(forecast_df["time"])
                forecast_df = forecast_df[
                    (forecast_df["time"] >= now) & 
                    (forecast_df["time"] <= prediction_end)
                ]
                all_data.append(forecast_df)
                print(f"    ✓ Forecast: {len(forecast_df)} heures")
            else:
                print(f"    ✗ Forecast: structure invalide")
                
        except Exception as e:
            print(f"    ✗ Forecast: {e}")
    
    # ===== COMBINER LES DONNÉES =====
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True).sort_values("time")
        zone_data[zone_key] = combined_df
        success_count += 1
        print(f"  ✓ Total combiné: {len(combined_df)} heures")
        print(f"    Période: {combined_df['time'].iloc[0]} → {combined_df['time'].iloc[-1]}")
    else:
        error_count += 1
        print(f"  ✗ Aucune donnée récupérée")

print(f"\n{'═' * 80}")
print(f"RÉSUMÉ")
print(f"{'═' * 80}")
print(f"✓ Zones avec succès: {success_count}/{len(WEATHER_ZONES)}")
print(f"✗ Zones avec erreur: {error_count}/{len(WEATHER_ZONES)}")

if success_count > 0:
    print(f"\n✓ L'import des données météo fonctionne correctement!")
    print(f"  Toutes les variables nécessaires sont disponibles:")
    for var in HOURLY_VARIABLES:
        print(f"    - {var}")
else:
    print(f"\n✗ Aucune zone n'a pu être téléchargée - vérifier la connexion réseau")

In [0]:
# Afficher les données de toutes les zones
for zone_key, df in zone_data.items():
    print(f"\n{'=' * 80}")
    print(f"Zone: {zone_key.upper()}")
    print(f"{'=' * 80}")
    print(f"Nombre de lignes: {len(df)}")
    print(f"\nPremières lignes:")
    display(df.head(10))
    print(f"\nDernières lignes:")
    display(df.tail(10))
    print(f"\nStatistiques descriptives:")
    display(df.describe())